# Baseline Model Training

This notebook will be used to train and evaluete initial machine-learning models for the Board Game Rating Predictor project.

Goal is to create simple baseline models that predict `Rating Average` using the prepared modelling datasets created during feature engineering.

This notebook will:

- load prepared training and testing datasets
- train simple baseline model
- train initial regression model
- evaluate model performance
- compare model results
- document baseline modelling decisions

In [163]:
# Import libraries used throghout this notebook
import pandas as pd

# Import Path for creating file paths that work
# reliably across different operating systems.
from pathlib import Path

# Import baseline modelling toool
from sklearn.dummy import DummyRegressor
# LinearRegression -first real regression model we are training
from sklearn.linear_model import LinearRegression

In [164]:
# Define current notebook folder and project root
current_folder = Path.cwd()
project_root = current_folder.parent

print("Current folder;", current_folder)
print("Project root:", project_root)

Current folder; c:\Users\Zombie\Desktop\vscode-projects\board-game-rating-predictor\notebooks
Project root: c:\Users\Zombie\Desktop\vscode-projects\board-game-rating-predictor


## Load Prepared Modelling Datasets

The prepared training and testing datasets are loaded from the `data/processed` folder.

These files were created during the feature engineering notebook.

Loaded files are following:
- training features
- testing features
- training target values
- testing target values
- model feature names
- imputation summary

Because the feature files were already prepared and imputed, they should not contain missing values.

In [165]:
# Define paths to prepared modelling dataset files
processed_data_folder = project_root / "data" / "processed"

x_train_path = processed_data_folder / "x_train_prepared.csv"
x_test_path = processed_data_folder / "x_test_prepared.csv"
y_train_path = processed_data_folder / "y_train.csv"
y_test_path = processed_data_folder / "y_test.csv"

model_feature_names_path = processed_data_folder / "model_feature_names.csv"
imputation_summary_path = processed_data_folder / "imputation_summary.csv"

print("Processed data folder:", processed_data_folder)
print("Folder exists:", processed_data_folder.exists())

Processed data folder: c:\Users\Zombie\Desktop\vscode-projects\board-game-rating-predictor\data\processed
Folder exists: True


In [166]:
# Cofrim that all required prepared modelling files exist
prepared_file_paths = [
    x_train_path,
    x_test_path,
    y_train_path,
    y_test_path,
    model_feature_names_path,
    imputation_summary_path
]

for file_path in prepared_file_paths:
    print(file_path.name, "exists:", file_path.exists())

x_train_prepared.csv exists: True
x_test_prepared.csv exists: True
y_train.csv exists: True
y_test.csv exists: True
model_feature_names.csv exists: True
imputation_summary.csv exists: True


In [167]:
# Load prepared modelling datasets
X_train = pd.read_csv(x_train_path)
X_test = pd.read_csv(x_test_path)

y_train_data = pd.read_csv(y_train_path)
y_test_data = pd.read_csv(y_test_path)

model_feature_names = pd.read_csv(model_feature_names_path)
imputation_summary = pd.read_csv(imputation_summary_path)

print("Prepared modelling files loaded successfully.")

Prepared modelling files loaded successfully.


In [168]:
# Define target colum
target_column = "Rating Average"

# Convert target DataFrames into Series for modelling
y_train = y_train_data[target_column]
y_test = y_test_data[target_column]

print("y_train type:", type(y_train))
print("y_test type:", type(y_test))

y_train type: <class 'pandas.Series'>
y_test type: <class 'pandas.Series'>


In [169]:
# Check the shapes of the loaded modelling datasets
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)
print("model_feature_names shape:", model_feature_names.shape)
print("imputation_summary shape:", imputation_summary.shape)

X_train shape: (16273, 35)
X_test shape: (4069, 35)
y_train shape: (16273,)
y_test shape: (4069,)
model_feature_names shape: (35, 1)
imputation_summary shape: (6, 2)


In [170]:
# Verify if loaded feature columns match saved modl feature names
expected_feature_names = model_feature_names["Feature"].tolist()

x_train_columns_match = X_train.columns.tolist() == expected_feature_names
x_test_columns_match = X_test.columns.tolist() == expected_feature_names

print("X_train columns match saved feature names:", x_train_columns_match)
print("X_test columns match saved feature names:", x_test_columns_match)

X_train columns match saved feature names: True
X_test columns match saved feature names: True


In [171]:
# Check missing values in the loaded prepared datasets
print("Missing values in X_train:", X_train.isnull().sum().sum())
print("Missing values in X_test:", X_test.isnull().sum().sum())
print("Missing values in y_train:", y_train.isnull().sum())
print("Missing values in y_test:", y_test.isnull().sum())

Missing values in X_train: 0
Missing values in X_test: 0
Missing values in y_train: 0
Missing values in y_test: 0


In [172]:
# Preview loaded training features
X_train.head()

,Year Published,Min Players,Min Age,Complexity Average,Max Players Log,Play Time Log,Year Published Missing,Min Players Missing,Max Players Missing,Play Time Missing,...,Mechanic: Area Movement,Mechanic: Simultaneous Action Selection,Domain: Wargames,Domain: Strategy Games,Domain: Family Games,Domain: Thematic Games,Domain: Abstract Games,Domain: Children's Games,Domain: Party Games,Domain: Customizable Games
0,2017.0,2.0,10.0,2.00,1.791759,4.110874,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,2013.0,2.0,11.0,1.43,1.945910,3.828641,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,2019.0,2.0,8.0,2.00,1.098612,3.433987,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,2004.0,2.0,12.0,3.00,1.098612,7.090910,0,0,0,0,...,1,0,1,0,0,0,0,0,0,0
4,2010.0,2.0,8.0,1.40,1.945910,2.772589,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [173]:
# Preview the loaded training target values
y_train.head()

0    5.86
1    4.53
2    7.33
3    7.60
4    5.97
Name: Rating Average, dtype: float64

### Prepared Modelling Dataset Load Result

Prepared modelling datasets were loaded successfully from the `data/processed` folder.

Training feature dataset contains 16,273 rows and 35 feature columns.

Testing feature dataset contains 4,069 rows and 35 feature columns.

Target values were loaded and converted into one-dimensional Series objects for modelling.

Loaded feature columns match the saved model feature names.

Loaded prepared datasets contain no missing values and and ready for baseline model training.

## Train Baseline Dummy Model

A baseline dummy model is trained before building more advanced machine-learning models.

This baseline model uses the mean `Rating Average` value from the training data and predicts that same value for every game.

Dummy model is not accurate (but it is not expected to be). Its purpose is to provide a simple comparison point.

Future models should perform better than this baseline.

In [174]:
# Create a baseline dummy regression model
baseline_dummy_model = DummyRegressor(strategy="mean")

# Train the dummy model on the training data
baseline_dummy_model.fit(X_train, y_train)

print("Baseline dummy model trained successfully.")
print("Baseline startegy:", baseline_dummy_model.strategy)

Baseline dummy model trained successfully.
Baseline startegy: mean


In [175]:
# Dummy model predicts training target mean for every row
baseline_prediction_value = baseline_dummy_model.constant_.item()

print("Training target mean:", y_train.mean())
print("Baseline model constant prediction:", baseline_prediction_value)

Training target mean: 6.403027714619308
Baseline model constant prediction: 6.403027714619308


In [176]:
# Create baseline predictions for training and testing feature sets
baseline_train_predictions = baseline_dummy_model.predict(X_train)
baseline_test_predictions = baseline_dummy_model.predict(X_test)

print("Baseline train predictions shape:", baseline_train_predictions.shape)
print("Baseline test predictions shape:", baseline_test_predictions.shape)

Baseline train predictions shape: (16273,)
Baseline test predictions shape: (4069,)


In [177]:
# Check how many unique prediction values the dummy model has created
unique_baseline_train_predictions = pd.Series(
    baseline_train_predictions
).nunique()

unique_baseline_test_predictions = pd.Series(
    baseline_test_predictions
).nunique()

print("Unique baseline train prediction values:", unique_baseline_train_predictions)
print("Unique baseline test prediction values:", unique_baseline_test_predictions)

Unique baseline train prediction values: 1
Unique baseline test prediction values: 1


In [178]:
# Preview actual testing ratings beside baseline predictions
baseline_prediction_preview = pd.DataFrame(
    {
        "Actual Rating": y_test.head(10).values,
        "Baseline Prediction": baseline_test_predictions[:10]
    }
)

baseline_prediction_preview

,Actual Rating,Baseline Prediction
0,6.23,6.403028
1,7.02,6.403028
2,6.96,6.403028
3,7.07,6.403028
4,8.88,6.403028
5,7.68,6.403028
6,6.88,6.403028
7,6.17,6.403028
8,7.53,6.403028
9,6.15,6.403028


### Baseline Dummy Model Result

A baseline dummy regression model was trained using training dataset.

The model uses the mean `Rating Average` value from training target values.

It predicts the same rating value for every board game.

Model is purposly made simple and will be used as a comparison point for future regression models.

A useful machine-learning model should perform better than this dummy baseline.

## Train Linear Regression Model

A Linear Regression model is trained as the first real machine-learning model.

Unlike the dummy baseline model we created before, Linear Regression uses the prepared feature columns to make predictions.

The purpose of this model is to check whether a simple regression model can learn useful patterns from the board game features and perform better than the dummy baseline.

In [179]:
# Create Linear Regression model
linear_regression_model = LinearRegression()

# Train model using the prepared training features and training target
linear_regression_model.fit(X_train, y_train)

print("Linear Regression model trained successfully (Party time!!!)")

Linear Regression model trained successfully (Party time!!!)


In [180]:
# Check the trained Linear Regression model structure
# Intercept is the model’s starting prediction before feature effects added
print("Model intercept:", linear_regression_model.intercept_)
#  Number of coefficients should match the number of feature columns
print("Number of model coefficients:", len(linear_regression_model.coef_))
print("Number of training features:", X_train.shape[1])

Model intercept: 4.63285114944448
Number of model coefficients: 35
Number of training features: 35


In [181]:
# This creates predictions that will be evaluate later. 
# We create predictions for both training and tetsing data
# We later compare how the mode performs on seen VS unseen data
linear_train_predictions = linear_regression_model.predict(X_train)
linear_test_predictions = linear_regression_model.predict(X_test)

print("Linear Regression train predictions shape:", linear_train_predictions.shape)
print("Linear Regression test predictions shape:", linear_test_predictions.shape)

Linear Regression train predictions shape: (16273,)
Linear Regression test predictions shape: (4069,)


In [182]:
# Review the distribution of Linear Regression test predictions
# - helps to quickly check wheather the predictions look reasonable
linear_test_prediction_summary = pd.Series(
    linear_test_predictions,
    name="Linear Regression Test Predictions"
).describe()

linear_test_prediction_summary

count    4069.000000
mean        6.401083
std         0.538451
min         4.198142
25%         6.025324
50%         6.394544
75%         6.765077
max         8.116889
Name: Linear Regression Test Predictions, dtype: float64

In [ ]:
# Preview actual testing ratings beside Linear Regression predictions
# -> human-readable first look at predictions
linear_prediction_preview = pd.DataFrame(
    {
        "Actual Rating": y_test.head(10).values,
        "Linear Regression Prediction": linear_test_predictions[:10]
    }
)

linear_prediction_preview

,Actual Rating,Linear Regression Prediction
0,6.23,6.370959
1,7.02,6.276297
2,6.96,7.099881
3,7.07,6.450589
4,8.88,7.706117
5,7.68,6.606565
6,6.88,7.144250
7,6.17,6.573564
8,7.53,6.496619
9,6.15,6.002259


### Linear Regression Model Result

A Lineart Regression model was trained using the prepared training dataset.

The model used all 35 prepared feature columns to learn relationships between board game attributes and `Rating Average`.

Predictions were created for both the training and testing datasets.

We are not ealuating model predictions against dummy baseline just yet (in next step)